In [ ]:
import os
import sys
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import SimpleITK as sitk
import pandas as pd
import torch
import torch.nn.functional as F

sys.path.append('/work/jprieto/data/remote/EGower/jprieto/source/trachoma/src/py/')
from nets.segmentation import TTUNet


In [ ]:
# df_train = pd.read_csv('/CMF/data/lumargot/trachoma/csv_mtss_pret/csv_updated/mtss_pret_combined_train.csv')
# df_train_seg = df_train[['filename']].drop_duplicates().reset_index(drop=True)
# df_train_seg['seg'] = df_train_seg['filename'].apply(lambda x: x.replace('/img/', '/seg/').replace('.jpg', '.nrrd'))
# df_train_seg = df_train_seg[df_train_seg.apply(lambda x: os.path.exists(os.path.join(mount_point, x['seg'])) and os.path.exists(os.path.join(mount_point, x['filename'])), axis=1)].reset_index(drop=True)
# df_train_seg.to_csv('/CMF/data/lumargot/trachoma/csv_mtss_pret/csv_updated/mtss_pret_seg_train.csv', index=False)

# df_test = pd.read_csv('/MEDUSA_STOR/jprieto/EGower/CSV_files/mtss_pret_combined_test.csv')
# df_test_seg = df_test[['filename']].drop_duplicates().reset_index(drop=True)
# df_test_seg['seg'] = df_test_seg['filename'].apply(lambda x: x.replace('/img/', '/seg/').replace('.jpg', '.nrrd'))
# df_test_seg = df_test_seg[df_test_seg.apply(lambda x: os.path.exists(os.path.join(mount_point, x['seg'])) and os.path.exists(os.path.join(mount_point, x['filename'])), axis=1)].reset_index(drop=True)
# df_test_seg.to_csv('/MEDUSA_STOR/jprieto/EGower/CSV_files/mtss_pret_seg_test.csv', index=False)

In [ ]:
# df_mtss = pd.read_csv('/MEDUSA_STOR/jprieto/EGower/CSV_files/mtss_pret_seg_train.csv')
# df_tt_seg = pd.read_csv('/MEDUSA_STOR/jprieto/EGower/CSV_files/trachoma_bsl_mtss_besrat_field_train_202208_seg_stack.csv')
# df_seg_train = pd.concat([df_tt_seg[['image', 'seg']].rename(columns={'image': 'img'}), df_mtss.rename(columns={'filename': 'img'})]).drop_duplicates().reset_index(drop=True)
# df_seg_train.to_csv('/MEDUSA_STOR/jprieto/EGower/CSV_files/full_seg_train.csv', index=False)

In [ ]:
# df_mtss = pd.read_csv('/MEDUSA_STOR/jprieto/EGower/CSV_files/mtss_pret_seg_test.csv')
# df_tt_seg = pd.read_csv('/MEDUSA_STOR/jprieto/EGower/CSV_files/trachoma_bsl_mtss_besrat_field_test_202208.csv')
# df_seg_test = pd.concat([df_tt_seg[['image', 'seg']].rename(columns={'image': 'img'}), df_mtss.rename(columns={'filename': 'img'})]).drop_duplicates().reset_index(drop=True)
# df_seg_test.to_csv('/MEDUSA_STOR/jprieto/EGower/CSV_files/full_seg_test.csv', index=False)

In [ ]:
def export(model, model_fn):
    input_tensor = torch.rand((1, 3, 512, 512), dtype=torch.float32).cuda()

    torch.onnx.export(
        model,                  # model to export
        (input_tensor,),        # inputs of the model,
        model_fn,               # filename of the ONNX model
        input_names=["input"],  # Rename inputs for the ONNX model
        dynamo=True             # True or False to select the exporter to use
    )

def export_torchscript(model, model_fn):
    input_tensor = torch.rand((1, 3, 512, 512), dtype=torch.float32).cuda()
    traced_script_module = torch.jit.trace(model, input_tensor)
    traced_script_module.save(model_fn)

In [ ]:
model_fn = '/MEDUSA_STOR/jprieto/EGower/train_output/segmentation/ttunet/full_seg/v5.3/epoch=189-val_loss=0.17.ckpt'
# model = TTUNet.load_from_checkpoint(model_fn, weights_only=False)
# export(model, model_fn.replace('.ckpt', '.onnx'))
# export_torchscript(model.model, model_fn.replace('.ckpt', '.pt'))

model = torch.jit.load(model_fn.replace('.ckpt', '.pt'))
model.eval()
model.cuda()

model.eval()
model.cuda()

In [ ]:


def _resize_with_grid_sample(
    x: torch.Tensor,
    out_hw: tuple[int, int],
    mode: str,
    align_corners: bool = False,
) -> torch.Tensor:
    """
    x: (N,C,H,W) tensor
    out_hw: (H_out, W_out)
    mode: 'bilinear' for images/logits, 'nearest' for label maps
    """
    assert x.ndim == 4, f"Expected (N,C,H,W), got {x.shape}"

    mesh_grid_params = [torch.arange(start=-1.0, end=1.0, step=(2.0/s), device=x.device) for s in out_hw]
    mesh_grid = torch.stack(torch.meshgrid(mesh_grid_params, indexing='xy'), dim=-1).to(torch.float32).unsqueeze(0)

    y = F.grid_sample(
        x, mesh_grid,
        mode=mode,
        padding_mode="zeros",
        align_corners=align_corners,
    )
    return y

@torch.no_grad()
def run_segmentation_resize_roundtrip(
    model,
    img: torch.Tensor,
    *,
    in_hw: tuple[int, int] = (512, 512),
    align_corners: bool = False,
    return_logits: bool = True,
):
    """
    img: (N,C,H,W)
    Returns:
      - logits_back: (N,K,H,W) if return_logits=True
      - or labels_back: (N,1,H,W) long tensor if return_logits=False
    """
    assert img.ndim == 4, f"Expected (N,C,H,W), got {img.shape}"
    n, c, h0, w0 = img.shape

    # 1) resize to model input
    img_512 = _resize_with_grid_sample(
        img, in_hw, mode="nearest", align_corners=align_corners
    )

    # 2) run model (expects 512x512)
    logits_512 = model(img_512)  # (N,K,512,512) typically

    # 3) resize back to original size
    logits_back = _resize_with_grid_sample(
        logits_512.float(), (w0, h0), mode="nearest", align_corners=align_corners
    )

    if return_logits:
        return logits_back

    # If you want discrete labels, do argmax at original resolution
    labels_back = torch.argmax(logits_back, dim=1, keepdim=True).to(torch.long)  # (N,1,H,W)
    return labels_back, img_512, logits_512


In [ ]:
mount_point = '/CMF/data/lumargot/trachoma/'
# img = 'B images one eye/img/1183_B-RIGHT.jpg'
# seg = 'B images one eye/seg/1183_B-RIGHT.nrrd'
img = '/MEDUSA_STOR/jprieto/EGower/PoPP_Data/mtss/img/11398_12Intra_od_postop.jpg'

img_np = sitk.GetArrayFromImage(sitk.ReadImage(os.path.join(mount_point, img)))
# seg_orig_np = sitk.GetArrayFromImage(sitk.ReadImage(os.path.join(mount_point, seg)))


In [ ]:

seg_t, img_512, logits_512 = run_segmentation_resize_roundtrip(model, torch.from_numpy(img_np).permute(2,0,1).unsqueeze(0).float().cuda()/255.0, return_logits=False)

In [ ]:
def plot_image(img, seg):
    """
    img: H x W x 3
    seg: H x W
    """

    assert img.shape[:2] == seg.shape

    fig = go.Figure()

    # Base RGB image (trace 0)
    fig.add_trace(
        go.Image(z=img)
    )

    # Overlay (trace 1)
    fig.add_trace(
        go.Heatmap(
            z=seg,
            colorscale="Jet",
            opacity=0.4,
            zmin=seg.min(),
            zmax=seg.max(),
            showscale=True,
            hoverinfo="skip"
        )
    )

    # ---- Opacity slider ----
    opacities = np.linspace(0, 1, 11)

    steps = [
        dict(
            method="restyle",
            args=[{"opacity": [None, o]}, [0, 1]],
            label=f"{o:.1f}"
        )
        for o in opacities
    ]

    fig.update_layout(
        sliders=[dict(
            active=4,
            currentvalue={"prefix": "Overlay opacity: "},
            pad={"t": 30},
            steps=steps
        )],
        xaxis=dict(
            showticklabels=False,
            range=[0, img.shape[1]]
        ),
        yaxis=dict(
            showticklabels=False,
            range=[img.shape[0], 0],
            scaleanchor="x"
        ),
        margin=dict(l=0, r=0, t=20, b=0),
    )

    fig.show()

In [ ]:
img_512.shape

In [ ]:
print(img_512.squeeze().permute(1,2,0).cpu().numpy().shape, logits_512.argmax(dim=1).squeeze().cpu().numpy().shape)

img_512_np = img_512.squeeze().permute(1,2,0).cpu().numpy()*255.0
img_512_np = img_512_np.astype(np.uint8)

plot_image(img_512_np, logits_512.argmax(dim=1).squeeze().cpu().numpy())
